## 0. Environment Verification

In [1]:
# Do NOT set CUDA_VISIBLE_DEVICES here — TF will be restricted later
import sys, os, numpy as np, torch, cv2, glob, shutil
from ultralytics import YOLO

print('=' * 55)
print(f'  Python  : {sys.version.split()[0]}')
print(f'  NumPy   : {np.__version__}')
print(f'  PyTorch : {torch.__version__}')
print(f'  OpenCV  : {cv2.__version__}')
if torch.cuda.is_available():
    print(f'  GPU     : {torch.cuda.get_device_name(0)} ✅')
    print(f'  VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print('  GPU     : ❌ Not available')
print('=' * 55)
print('  ✅ Ready!')

  Python  : 3.10.19
  NumPy   : 1.26.4
  PyTorch : 2.7.1+cu118
  OpenCV  : 4.12.0
  GPU     : NVIDIA GeForce RTX 4060 Laptop GPU ✅
  VRAM    : 8.0 GB
  ✅ Ready!


## 1. Configuration

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
OBB_ROOT        = r'D:\leaf.v1-yolo_leaf1.yolov5-obb'
PLANTDOC_ROOT   = r'D:\Fine Tuning'
MERGED_ROOT     = r'D:\Merged_OBB_Dataset'
CLASSIFIER_PATH = r'D:\Leaf dataset 2\outputs\CustomCNN\CustomCNN_best.keras'
MERGED_OUTPUT   = os.path.join(MERGED_ROOT, 'outputs')
MERGED_YAML     = os.path.join(MERGED_ROOT, 'data.yaml')

# ── Find existing OBB weights ──────────────────────────────────────────────
obb_candidates = glob.glob(os.path.join(OBB_ROOT, 'outputs', '**', 'best.pt'), recursive=True)
if obb_candidates:
    OBB_WEIGHTS = sorted(obb_candidates)[-1]
    print(f'✅ OBB weights: {OBB_WEIGHTS}')
else:
    raise FileNotFoundError(f'No OBB weights in {OBB_ROOT}\\outputs')

# ── OBB Classes ───────────────────────────────────────────────────────────
OBB_CLASSES   = ['Blight_a','Blight_b','Blight_c','Danger_a','Danger_b','Healthy_a','Healthy_b']
OBB_CLASS_MAP = {name: idx for idx, name in enumerate(OBB_CLASSES)}

# ── PlantDoc index → name ─────────────────────────────────────────────────
PLANTDOC_IDX_TO_NAME = {
    0:'Cherry leaf', 1:'Peach leaf', 2:'Corn leaf blight', 3:'Apple rust leaf',
    4:'Potato leaf late blight', 5:'Strawberry leaf', 6:'Corn rust leaf',
    7:'Tomato leaf late blight', 8:'Tomato mold leaf', 9:'Potato leaf early blight',
    10:'Apple leaf', 11:'Tomato leaf yellow virus', 12:'Blueberry leaf',
    13:'Tomato leaf mosaic virus', 14:'Raspberry leaf', 15:'Tomato leaf bacterial spot',
    16:'Squash Powdery mildew leaf', 17:'grape leaf', 18:'Corn Gray leaf spot',
    19:'Tomato Early blight leaf', 20:'Apple Scab Leaf', 21:'Tomato Septoria leaf spot',
    22:'Tomato leaf', 23:'Soyabean leaf', 24:'Bell_pepper leaf spot',
    25:'Bell_pepper leaf', 26:'grape leaf black rot', 27:'Potato leaf',
    28:'Tomato two spotted spider mites leaf',
}

# ── PlantDoc 29 → OBB 7 severity mapping ─────────────────────────────────
PLANTDOC_TO_OBB = {
    'Apple leaf':'Healthy_a', 'Cherry leaf':'Healthy_a', 'Peach leaf':'Healthy_a',
    'Strawberry leaf':'Healthy_b', 'Blueberry leaf':'Healthy_a', 'Raspberry leaf':'Healthy_a',
    'Soyabean leaf':'Healthy_a', 'Tomato leaf':'Healthy_a', 'Potato leaf':'Healthy_a',
    'grape leaf':'Healthy_a', 'Bell_pepper leaf':'Healthy_a',
    'Apple rust leaf':'Danger_a', 'Apple Scab Leaf':'Danger_b',
    'Corn rust leaf':'Danger_a', 'Corn Gray leaf spot':'Danger_a',
    'Tomato leaf bacterial spot':'Danger_a', 'Tomato leaf yellow virus':'Danger_b',
    'Tomato leaf mosaic virus':'Danger_b', 'Potato leaf early blight':'Danger_b',
    'Bell_pepper leaf spot':'Danger_a', 'Squash Powdery mildew leaf':'Danger_b',
    'Corn leaf blight':'Blight_a', 'Tomato Early blight leaf':'Blight_a',
    'Tomato Septoria leaf spot':'Blight_a', 'Tomato mold leaf':'Blight_b',
    'grape leaf black rot':'Blight_b', 'Tomato two spotted spider mites leaf':'Blight_a',
    'Tomato leaf late blight':'Blight_c', 'Potato leaf late blight':'Blight_c',
}

# ── Classifier Classes (38) ───────────────────────────────────────────────
CLASSIFIER_CLASSES = [
    'Apple___Apple_scab','Apple___Black_rot','Apple___Cedar_apple_rust','Apple___healthy',
    'Blueberry___healthy','Cherry_(including_sour)___Powdery_mildew','Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot','Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight','Corn_(maize)___healthy',
    'Grape___Black_rot','Grape___Esca_(Black_Measles)','Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy','Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot','Peach___healthy','Pepper,_bell___Bacterial_spot','Pepper,_bell___healthy',
    'Potato___Early_blight','Potato___Late_blight','Potato___healthy','Raspberry___healthy',
    'Soybean___healthy','Squash___Powdery_mildew','Strawberry___Leaf_scorch','Strawberry___healthy',
    'Tomato___Bacterial_spot','Tomato___Early_blight','Tomato___Late_blight','Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot','Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot','Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus','Tomato___healthy'
]

# ── Severity display ──────────────────────────────────────────────────────
SEVERITY_MAP = {
    'Healthy_a':'🟢 Healthy (Confirmed)', 'Healthy_b':'🟡 Healthy (Minor marks)',
    'Danger_a':'🟠 Warning — Early Stage', 'Danger_b':'🟠 Caution — Moderate',
    'Blight_a':'🔴 Blight — Mild', 'Blight_b':'🔴 Blight — Moderate',
    'Blight_c':'🔴 Blight — SEVERE ⚠️', 'Unknown':'⚪ Unknown',
}
SEVERITY_COLORS = {
    'Healthy_a':(0,200,0), 'Healthy_b':(0,180,80),
    'Danger_a':(255,165,0), 'Danger_b':(255,120,0),
    'Blight_a':(220,50,50), 'Blight_b':(180,20,20),
    'Blight_c':(130,0,0), 'Unknown':(128,128,128),
}

os.makedirs(MERGED_ROOT, exist_ok=True)
os.makedirs(MERGED_OUTPUT, exist_ok=True)
print('✅ Configuration loaded')
print(f'   OBB weights: {OBB_WEIGHTS}')

✅ OBB weights: D:\leaf.v1-yolo_leaf1.yolov5-obb\outputs\yolov8m_obb\weights\best.pt
✅ Configuration loaded
   OBB weights: D:\leaf.v1-yolo_leaf1.yolov5-obb\outputs\yolov8m_obb\weights\best.pt


## 2. Convert PlantDoc Labels → OBB Format

In [3]:
from tqdm import tqdm

def yolo_to_obb(cx, cy, w, h):
    x1,y1 = cx-w/2, cy-h/2
    x2,y2 = cx+w/2, cy-h/2
    x3,y3 = cx+w/2, cy+h/2
    x4,y4 = cx-w/2, cy+h/2
    return [max(0.,min(1.,c)) for c in [x1,y1,x2,y2,x3,y3,x4,y4]]

def convert_plantdoc_split(split):
    src_lbl = os.path.join(PLANTDOC_ROOT, 'labels', split)
    src_img = os.path.join(PLANTDOC_ROOT, 'images', split)
    dst_lbl = os.path.join(MERGED_ROOT, split, 'labels')
    dst_img = os.path.join(MERGED_ROOT, split, 'images')
    os.makedirs(dst_lbl, exist_ok=True)
    os.makedirs(dst_img, exist_ok=True)

    converted, skipped = 0, 0
    for lf in tqdm(glob.glob(os.path.join(src_lbl, '*.txt')), desc=f'PlantDoc {split}'):
        stem, out = os.path.splitext(os.path.basename(lf))[0], []
        with open(lf) as f:
            for line in f:
                p = line.strip().split()
                if len(p) != 5: continue
                name = PLANTDOC_IDX_TO_NAME.get(int(p[0]))
                if not name: continue
                sev  = PLANTDOC_TO_OBB.get(name, 'Danger_a')
                idx  = OBB_CLASS_MAP[sev]
                c    = yolo_to_obb(*map(float, p[1:]))
                out.append(f'{idx} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f} {c[3]:.6f} {c[4]:.6f} {c[5]:.6f} {c[6]:.6f} {c[7]:.6f}')
        with open(os.path.join(dst_lbl, os.path.basename(lf)), 'w') as f:
            f.write('\n'.join(out))
        for ext in ['.jpg','.JPG','.jpeg','.png','.PNG']:
            s = os.path.join(src_img, stem+ext)
            if os.path.exists(s):
                shutil.copy2(s, os.path.join(dst_img, stem+ext))
                converted += 1; break
        else:
            skipped += 1
    print(f'  ✅ {split}: converted={converted} skipped={skipped}')

print('Converting PlantDoc → OBB format...')
convert_plantdoc_split('train')
convert_plantdoc_split('val')
print('✅ Done!')

Converting PlantDoc → OBB format...


PlantDoc train: 100%|██████████| 2335/2335 [01:10<00:00, 33.31it/s]


  ✅ train: converted=2335 skipped=0


PlantDoc val: 100%|██████████| 236/236 [00:06<00:00, 37.90it/s]

  ✅ val: converted=236 skipped=0
✅ Done!


## 3. Copy Original OBB Dataset

In [4]:
def copy_obb_split(split):
    obb_split = 'valid' if split == 'val' else split
    src_img   = os.path.join(OBB_ROOT, obb_split, 'images')
    src_lbl   = os.path.join(OBB_ROOT, obb_split, 'labels')
    dst_img   = os.path.join(MERGED_ROOT, split, 'images')
    dst_lbl   = os.path.join(MERGED_ROOT, split, 'labels')
    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    images = []
    for ext in ['*.jpg','*.JPG','*.jpeg','*.png','*.PNG']:
        images += glob.glob(os.path.join(src_img, ext))

    copied = 0
    for ip in tqdm(images, desc=f'OBB {split}'):
        stem = os.path.splitext(os.path.basename(ip))[0]
        shutil.copy2(ip, os.path.join(dst_img, 'obb_'+os.path.basename(ip)))
        lp = os.path.join(src_lbl, stem+'.txt')
        dl = os.path.join(dst_lbl, 'obb_'+stem+'.txt')
        if os.path.exists(lp): shutil.copy2(lp, dl)
        else: open(dl,'w').close()
        copied += 1
    print(f'  ✅ {split}: {copied} images copied')

print('Copying original OBB dataset...')
copy_obb_split('train')
copy_obb_split('val')
print('✅ Done!')

Copying original OBB dataset...


OBB train: 100%|██████████| 23644/23644 [07:41<00:00, 51.21it/s] 


  ✅ train: 23644 images copied


OBB val: 100%|██████████| 522/522 [00:08<00:00, 59.77it/s] 

  ✅ val: 522 images copied
✅ Done!


## 4. Verify Merged Dataset

In [5]:
print('='*60)
print('  Merged Dataset Verification')
print('='*60)
for split in ['train','val']:
    imgs, lbls = [], glob.glob(os.path.join(MERGED_ROOT, split, 'labels', '*.txt'))
    for ext in ['*.jpg','*.JPG','*.jpeg','*.png','*.PNG']:
        imgs += glob.glob(os.path.join(MERGED_ROOT, split, 'images', ext))
    total_boxes  = 0
    class_counts = {i:0 for i in range(7)}
    for lf in lbls:
        with open(lf) as f:
            for line in f:
                p = line.strip().split()
                if len(p)==9:
                    total_boxes += 1
                    class_counts[int(p[0])] += 1
    pd_imgs = len([f for f in imgs if 'obb_' not in os.path.basename(f)])
    ob_imgs = len([f for f in imgs if 'obb_' in os.path.basename(f)])
    print(f'\n📁 {split}:')
    print(f'   Total={len(imgs)} | PlantDoc={pd_imgs} | OBB={ob_imgs} | Boxes={total_boxes}')
    mx = max(class_counts.values()) or 1
    for i,n in enumerate(OBB_CLASSES):
        bar = '█'*int(class_counts[i]/mx*25)
        print(f'   {n:<12} {class_counts[i]:>6}  {bar}')

# Validate sample label
for lf in glob.glob(os.path.join(MERGED_ROOT,'train','labels','*.txt')):
    with open(lf) as f: content = f.read().strip()
    if content:
        line = content.splitlines()[0]
        cols = len(line.split())
        print(f'\n📋 Sample: {line}')
        print(f'   Cols={cols} — {"✅ Valid OBB" if cols==9 else "❌ Invalid"}')
        break
print('\n✅ Verification complete!')

  Merged Dataset Verification

📁 train:
   Total=28314 | PlantDoc=4670 | OBB=23644 | Boxes=19104
   Blight_a       2766  ███████████████
   Blight_b       2304  █████████████
   Blight_c       2280  █████████████
   Danger_a       3021  █████████████████
   Danger_b       4374  █████████████████████████
   Healthy_a      3851  ██████████████████████
   Healthy_b       508  ██

📁 val:
   Total=994 | PlantDoc=472 | OBB=522 | Boxes=666
   Blight_a         86  ████████████
   Blight_b         67  █████████
   Blight_c         64  █████████
   Danger_a         94  █████████████
   Danger_b        171  █████████████████████████
   Healthy_a       152  ██████████████████████
   Healthy_b        32  ████

📋 Sample: 3 1.000000 0.000000 0.035937 0.000000 0.035938 1.000000 1.000000 1.000000
   Cols=9 — ✅ Valid OBB

✅ Verification complete!


## 5. Create data.yaml

In [6]:
import yaml
yaml_content = {
    'path':'D:\\Merged_OBB_Dataset',
    'train':'train/images',
    'val':'val/images',
    'nc':7,
    'names':OBB_CLASSES
}
with open(MERGED_YAML,'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)
print('✅ data.yaml saved:')
with open(MERGED_YAML) as f: print(f.read())

✅ data.yaml saved:
names:
- Blight_a
- Blight_b
- Blight_c
- Danger_a
- Danger_b
- Healthy_a
- Healthy_b
nc: 7
path: D:\Merged_OBB_Dataset
train: train/images
val: val/images

